## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Nelder Mead
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()

In [4]:
competition_root = current_dir.parents[1]

In [5]:
# competition_root

In [6]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [7]:
train = pd.read_csv(competition_path/'data/train.csv')
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/'data/test.csv')
test_data = test[["id"]]

submit = pd.read_csv(competition_path/'data/sample_submission.csv')

In [8]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [9]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [10]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [11]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [12]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [13]:
'''
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])
'''

'\nX = X.drop(columns=["log_5cv"])\nX_test = X_test.drop(columns=["log_5cv"])\n'

In [14]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [15]:
num_models

37

In [16]:
# 2. Define the objective function with a penalty constraint
def objective(weights, predictions, y_true):
    # Standardize weights so they sum to 1 for the actual blend calculation
    # This helps Nelder-Mead evaluate valid combinations even during search
    sum_w = np.sum(weights)
    if sum_w == 0:
        normalized_weights = weights
    else:
        normalized_weights = weights / sum_w
    
    # Calculate the blended prediction
    blend = np.tensordot(normalized_weights, predictions, axes=((0), (1)))
    
    # Calculate our metric (Nelder-Mead MINIMIZES, so we do 1 - AUC)
    score = 1.0 - roc_auc_score(y_true, blend)
    
    # Penalty: If weights go negative, add a heavy penalty to steer the optimizer away
    penalty = 0
    if np.any(weights < 0):
        penalty += 10.0 * np.abs(np.min(weights))

    l1_lambda = 0.001
    penalty += l1_lambda * np.sum(np.abs(normalized_weights))
        
    return score + penalty

In [17]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [18]:
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)

In [19]:
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)

In [20]:
best_weights

array([0.0269528 , 0.02156026, 0.00080465, 0.00035708, 0.00984898,
       0.03916626, 0.02648417, 0.02695662, 0.0160753 , 0.03873788,
       0.03228974, 0.0254872 , 0.03893389, 0.02256783, 0.03945175,
       0.0519706 , 0.01430672, 0.00023746, 0.03413681, 0.03054112,
       0.00023136, 0.02261691, 0.0459486 , 0.03807868, 0.02842083,
       0.04000594, 0.0266227 , 0.02406212, 0.03438888, 0.0324829 ,
       0.0356989 , 0.02968216, 0.0252227 , 0.02600799, 0.04592672,
       0.03491407, 0.01282141])

In [21]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")

Optimization Successful: False
Optimized Blending Weights: [0.027  0.0216 0.0008 0.0004 0.0098 0.0392 0.0265 0.027  0.0161 0.0387
 0.0323 0.0255 0.0389 0.0226 0.0395 0.052  0.0143 0.0002 0.0341 0.0305
 0.0002 0.0226 0.0459 0.0381 0.0284 0.04   0.0266 0.0241 0.0344 0.0325
 0.0357 0.0297 0.0252 0.026  0.0459 0.0349 0.0128]
Best ROC-AUC Score: 0.9453


In [22]:
# Compute the final blended predictions
final_blend = np.dot(X_test.values, best_weights)

In [23]:
submit["is_fraud"] = final_blend

In [24]:
submit.to_csv(competition_path/"Blending Nelder Mead v7.csv",index=False)

In [45]:
# Drop high correlated variable

In [46]:
'''
# 1. Calculate the correlation matrix (if not already done)
corr_matrix = X.corr().abs()

# 2. Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
'''

'\n# 1. Calculate the correlation matrix (if not already done)\ncorr_matrix = X.corr().abs()\n\n# 2. Select the upper triangle of the correlation matrix\nupper_tri = corr_matrix.where(\n    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)\n)\n'

In [47]:
'''
# 3. Find features with correlation greater than the threshold (e.g., 0.95)
threshold = 0.97
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
'''

'\n# 3. Find features with correlation greater than the threshold (e.g., 0.95)\nthreshold = 0.97\nto_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]\n'

In [48]:
'''
# 4. Drop the features from your training and test sets
X_reduced = X.drop(columns=to_drop)
X_test_reduced = X_test.drop(columns=to_drop)
'''

'\n# 4. Drop the features from your training and test sets\nX_reduced = X.drop(columns=to_drop)\nX_test_reduced = X_test.drop(columns=to_drop)\n'

In [49]:
'''
oof_matrix = X_reduced.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]
'''

'\noof_matrix = X_reduced.values \ny_true_array = y.values\nnum_models = oof_matrix.shape[1]\n'

In [50]:
'''
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)
'''

'\n# 3. Setup and Run the Optimizer\n# Start with equal weights for all 3 models\ninit_weights = np.array([1/num_models]*num_models)\n'

In [51]:
'''
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X_reduced, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)
'''

"\n# Run Nelder-Mead\nres = minimize(\n    objective, \n    init_weights, \n    args=(X_reduced, y), \n    method='Nelder-Mead',\n    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}\n)\n"

In [52]:
'''
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)
'''

'\n# 4. Process and Normalize the Final Best Weights\nbest_raw_weights = res.x\nbest_weights = best_raw_weights / np.sum(best_raw_weights)\n'

In [53]:
'''
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")
'''

'\n# 5. Print Results\nprint("Optimization Successful:", res.success)\nprint(f"Optimized Blending Weights: {np.round(best_weights, 4)}")\nprint(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")\n'

In [54]:
'''
# Compute the final blended predictions
final_blend = np.dot(X_test_reduced.values, best_weights)
'''

'\n# Compute the final blended predictions\nfinal_blend = np.dot(X_test_reduced.values, best_weights)\n'

In [55]:
'''
submit["is_fraud"] = final_blend
submit.to_csv("Blending Nelder Mead v5.csv",index=False)
'''

'\nsubmit["is_fraud"] = final_blend\nsubmit.to_csv("Blending Nelder Mead v5.csv",index=False)\n'